# 21-03 · Ruch statków i wrogowie

Praktyka do sekcji [„Ruch statku i pojawianie się wrogów”](/pl/chapters/rozdzial-21/21-03-dvizhenie-vragi.html).

## Cel

Ogranicz ruch statku do krawędzi ekranu i zmuszaj wrogów do pojawiania się i opadania.

## O pętli gier w tym laptopie

W normalnym życiu `.py`pliku -plik, pętla gry to `while rabotaet:`to zależy od zdarzeń użytkownika (zamykanie okna, naciśnięcia klawiszy). W automatycznie wykonywanym laptopie nie ma nikogo, kto mógłby je tworzyć, więc tutaj uruchamiamy stałą liczbę klatek przez `for kadr in range(N):` — logika każdej pojedynczej klatki (ruch, kolizje, rysowanie) przy tym dokładnie taka sama, jak w prawdziwej grze z `projects/pygame/space-shooter/space_shooter.py`. Prędkość wciąż jest podana w pikselach na sekundę (px/s), a nie w pikselach na klatkę — ale aby wynik notatnika nie zależał od tego, jak szybko kod działa na konkretnym komputerze, `dt` tutaj nie jest zaczerpnięte z `clock.tick()`, ale jest ustalone z góry jako `1 / FPS`.

## Sprawa robocza

In [ ]:
import random

import pygame

SHIRINA, VYSOTA = 480, 720
FPS = 60

KORABL_SHIRINA, KORABL_VYSOTA = 44, 44
KORABL_SKOROST = 260.0   # px/s

PULYA_SHIRINA, PULYA_VYSOTA = 6, 18
PULYA_SKOROST = 560.0   # px/s

VRAG_SHIRINA, VRAG_VYSOTA = 32, 28
VRAG_SKOROST = 150.0                # px/s
INTERVAL_POYAVLENIYA_VRAGA = 0.75   # секунд между новыми врагами

BELYJ = (255, 255, 255)
CHERNYJ = (10, 10, 20)
ZELYONYJ = (80, 220, 120)
KRASNYJ = (230, 60, 60)
ZHYOLTYJ = (240, 220, 80)

pygame.init()
screen = pygame.display.set_mode((SHIRINA, VYSOTA))
pygame.display.set_caption("Космический шутер")
clock = pygame.time.Clock()
shrift = pygame.font.SysFont(None, 32)
shrift_bolshoj = pygame.font.SysFont(None, 64)


def novaya_igra():
    korabl = pygame.Rect(
        SHIRINA // 2 - KORABL_SHIRINA // 2,
        VYSOTA - KORABL_VYSOTA - 20,
        KORABL_SHIRINA,
        KORABL_VYSOTA,
    )
    return {
        "korabl": korabl,
        "korabl_x": float(korabl.x),
        "puli": [],
        "vragi": [],
        "schet": 0,
        "vremya_do_vraga": INTERVAL_POYAVLENIYA_VRAGA,
        "igra_okonchena": False,
    }


def obrabotat_klavishi(state, klavishi, dt):
    napravlenie = klavishi[pygame.K_RIGHT] - klavishi[pygame.K_LEFT]
    korabl_x = state["korabl_x"] + napravlenie * KORABL_SKOROST * dt
    state["korabl_x"] = max(0.0, min(korabl_x, SHIRINA - KORABL_SHIRINA))
    state["korabl"].x = round(state["korabl_x"])


def vystrelit(state):
    korabl = state["korabl"]
    pulya_rect = pygame.Rect(
        korabl.centerx - PULYA_SHIRINA // 2,
        korabl.top,
        PULYA_SHIRINA,
        PULYA_VYSOTA,
    )
    state["puli"].append({"rect": pulya_rect, "y": float(pulya_rect.y)})


def sozdat_vraga():
    x = random.randint(0, SHIRINA - VRAG_SHIRINA)
    return {"rect": pygame.Rect(x, -VRAG_VYSOTA, VRAG_SHIRINA, VRAG_VYSOTA), "y": float(-VRAG_VYSOTA)}


def obnovit_igru(state, dt):
    if state["igra_okonchena"]:
        return

    for pulya in state["puli"]:
        pulya["y"] -= PULYA_SKOROST * dt
        pulya["rect"].y = round(pulya["y"])
    state["puli"] = [p for p in state["puli"] if p["rect"].bottom > 0]

    state["vremya_do_vraga"] -= dt
    if state["vremya_do_vraga"] <= 0.0:
        state["vragi"].append(sozdat_vraga())
        state["vremya_do_vraga"] += INTERVAL_POYAVLENIYA_VRAGA

    for vrag in state["vragi"]:
        vrag["y"] += VRAG_SKOROST * dt
        vrag["rect"].y = round(vrag["y"])

    novye_puli = []
    novye_vragi = list(state["vragi"])
    for pulya in state["puli"]:
        popala = False
        for vrag in list(novye_vragi):
            if pulya["rect"].colliderect(vrag["rect"]):
                novye_vragi.remove(vrag)
                state["schet"] += 10
                popala = True
                break
        if not popala:
            novye_puli.append(pulya)
    state["puli"] = novye_puli
    state["vragi"] = novye_vragi

    for vrag in state["vragi"]:
        if vrag["rect"].bottom >= VYSOTA or vrag["rect"].colliderect(state["korabl"]):
            state["igra_okonchena"] = True
            break


def narisovat(state):
    screen.fill(CHERNYJ)
    pygame.draw.rect(screen, ZELYONYJ, state["korabl"])
    for pulya in state["puli"]:
        pygame.draw.rect(screen, ZHYOLTYJ, pulya["rect"])
    for vrag in state["vragi"]:
        pygame.draw.rect(screen, KRASNYJ, vrag["rect"])

    tablo = shrift.render(f"Счёт: {state['schet']}", True, BELYJ)
    screen.blit(tablo, (10, 10))

    if state["igra_okonchena"]:
        nadpis = shrift_bolshoj.render("ИГРА ОКОНЧЕНА", True, BELYJ)
        rect = nadpis.get_rect(center=(SHIRINA // 2, VYSOTA // 2))
        screen.blit(nadpis, rect)

    pygame.display.flip()

In [ ]:
state = novaya_igra()
dt = 1 / FPS

# symulacji: strzałka w prawo jest przytrzymywana przez 60 klatek z rzędu (60 * dt = 1,0 sekundy)
for kadr in range(60):
    klavishi = {pygame.K_LEFT: False, pygame.K_RIGHT: True}
    obrabotat_klavishi(state, klavishi, dt)
    obnovit_igru(state, dt)
    narisovat(state)
    clock.tick(FPS)

print("Корабль после 1 секунды движения вправо:", state["korabl"])
print("Врагов появилось:", len(state["vragi"]))

## Sprawdzenie wyniku

In [ ]:
assert state["korabl"].x == SHIRINA - KORABL_SHIRINA, "корабль должен упереться в правый край"
print("Верно: корабль остановился у правого края экрана, не выйдя за его пределы.")

assert len(state["vragi"]) >= 1, "за 1 секунду должен появиться хотя бы один враг (интервал 0.75 с)"
print(f"Найдено врагов: {len(state['vragi'])}")

## Podstawowa praktyka zadań ★

Uruchom kolejne 45 klatek (0,75 sekundy) bez ruchu statku i upewnij się, że pojawi się drugi wróg.

In [ ]:
for kadr in range(45):
    klavishi = {pygame.K_LEFT: False, pygame.K_RIGHT: False}
    obrabotat_klavishi(state, klavishi, dt)
    obnovit_igru(state, dt)
    narisovat(state)

print("Врагов теперь:", len(state["vragi"]))
assert len(state["vragi"]) >= 2
print("Верно: появился второй враг.")